## Section 2: Exploratory Data Analysis (EDA)

In [ ]:
# define the required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# load the airlines lookup table
airlines = pd.read_csv('Airlines.csv')
airlines.head()

In [ ]:
# load the flight dataset
df = pd.read_csv('Combined_Flights_2022.csv')
df.head()

In [ ]:
# the Airline column already contains full airline names, so no merge is needed
# drop Code and Description columns if they exist from a previous merge
df = df.drop(columns=[col for col in ["Code", "Description"] if col in df.columns])
df.head()

In [ ]:
# show the shape of the dataset
df.shape

In [ ]:
# show the data information
df.info()

In [ ]:
# show the data statistics
df.describe()

In [ ]:
# check for missing values
df.isnull().sum()

In [ ]:
# check for missing values in percentage
df.isnull().mean() * 100

## 2.1 Create Flight Status Column

In [ ]:
# check the Cancelled and Diverted columns
print(df['Cancelled'].value_counts())
print(df['Diverted'].value_counts())

In [ ]:
# create flight_status column from Cancelled, Diverted and DepDelayMinutes
def get_status(row):
    if row['Cancelled'] == True or row['Cancelled'] == 1:
        return 'Cancelled'
    elif row['Diverted'] == True or row['Diverted'] == 1:
        return 'Diverted'
    elif row['DepDelayMinutes'] > 15:
        return 'Delayed'
    else:
        return 'On Time'

df['flight_status'] = df.apply(get_status, axis=1)
df['flight_status'].value_counts()

## 2.2 Target Variable Distribution

In [ ]:
# visualize the flight status distribution
df['flight_status'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c', '#f39c12', '#3498db'], edgecolor='white')
plt.title('Flight Status Distribution')
plt.xlabel('Status')
plt.ylabel('Number of Flights')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# visualize the flight status as pie chart
df['flight_status'].value_counts().plot(kind='pie', autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c', '#f39c12', '#3498db'], startangle=90)
plt.title('Flight Status Distribution (%)')
plt.ylabel('')
plt.show()

## 2.3 Delay Analysis by Airline

In [ ]:
# calculate average delay by airline
airline_delay = df[df['flight_status'] == 'Delayed'].groupby('Airline')['DepDelayMinutes'].mean().sort_values(ascending=False).head(10)
airline_delay

In [ ]:
# visualize average delay by airline
airline_delay.plot(kind='barh', color='#e74c3c', edgecolor='white')
plt.title('Top 10 Airlines by Average Departure Delay (Minutes)')
plt.xlabel('Avg Delay (Minutes)')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# calculate delay rate by airline
delay_rate = df.groupby('Airline')['flight_status'].apply(
    lambda x: (x == 'Delayed').sum() / len(x) * 100
).sort_values(ascending=False).head(10)

delay_rate

In [ ]:
# visualize delay rate by airline
delay_rate.plot(kind='bar', color='#e67e22', edgecolor='white')
plt.title('Top 10 Airlines by Delay Rate (%)')
plt.xlabel('Airline')
plt.ylabel('Delay Rate (%)')
plt.xticks(rotation=30, ha='right')
plt.show()

## 2.4 Delay Trends by Time

In [ ]:
# extract departure hour from CRSDepTime
df['DepHour'] = df['CRSDepTime'].astype(str).str.zfill(4).str[:2].astype(int)
df['DepHour'] = df['DepHour'].clip(0, 23)
df['DepHour'].value_counts().sort_index()

In [ ]:
# calculate delay rate by hour
hourly = df.groupby('DepHour')['flight_status'].apply(lambda x: (x == 'Delayed').sum() / len(x) * 100).reset_index()
hourly.columns = ['Hour', 'Delay_Rate_%']
hourly

In [ ]:
# visualize delay rate by hour of day
plt.plot(hourly['Hour'], hourly['Delay_Rate_%'], marker='o', color='#e67e22', linewidth=2)
plt.fill_between(hourly['Hour'], hourly['Delay_Rate_%'], alpha=0.15, color='#e67e22')
plt.title('Flight Delay Rate by Hour of Day')
plt.xlabel('Departure Hour (24h)')
plt.ylabel('Delay Rate (%)')
plt.xticks(range(0, 24))
plt.show()

In [ ]:
# calculate delay rate by month
monthly = df.groupby('Month')['flight_status'].apply(
    lambda x: (x == 'Delayed').sum() / len(x) * 100
).reset_index()
monthly.columns = ['Month', 'Delay_Rate_%']
monthly

In [ ]:
# visualize delay rate by month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['Month_Name'] = monthly['Month'].apply(lambda x: month_names[x-1])

plt.bar(monthly['Month_Name'], monthly['Delay_Rate_%'], color=sns.color_palette('Spectral', len(monthly)), edgecolor='white')
plt.title('Flight Delay Rate by Month')
plt.xlabel('Month')
plt.ylabel('Delay Rate (%)')
plt.show()

In [ ]:
# calculate delay rate by day of week
daily = df.groupby('DayOfWeek')['flight_status'].apply(lambda x: (x == 'Delayed').sum() / len(x) * 100).reset_index()
daily.columns = ['DayOfWeek', 'Delay_Rate_%']
daily

In [ ]:
# visualize delay rate by day of week
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
plt.bar(day_names, daily['Delay_Rate_%'], color='#3498db', edgecolor='white')
plt.title('Flight Delay Rate by Day of Week')
plt.xlabel('Day')
plt.ylabel('Delay Rate (%)')
plt.show()

## 2.5 Delay Distribution

In [ ]:
# check the departure delay statistics for delayed flights only
df[df['DepDelayMinutes'] > 0]['DepDelayMinutes'].describe()

In [ ]:
# visualize the distribution of departure delay minutes
delayed_df = df[(df['DepDelayMinutes'] > 0) & (df['DepDelayMinutes'] < 300)]
plt.hist(delayed_df['DepDelayMinutes'], bins=60, color='#e74c3c', edgecolor='white', alpha=0.85)
plt.axvline(delayed_df['DepDelayMinutes'].median(), color='navy', linestyle='--', linewidth=2, label=f"Median: {delayed_df['DepDelayMinutes'].median():.0f} min")
plt.title('Distribution of Departure Delay Minutes')
plt.xlabel('Delay (Minutes)')
plt.ylabel('Number of Flights')
plt.legend()
plt.show()

## 2.6 Correlation Heatmap

In [ ]:
# select numerical columns for correlation
num_cols = ['DepDelayMinutes', 'ArrDelayMinutes', 'AirTime', 'Distance', 'TaxiOut', 'TaxiIn']
df[num_cols].corr()

In [ ]:
# visualize the correlation heatmap
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

## 2.7 Top Routes by Delay

In [ ]:
# create route column by combining origin and destination
df['Route'] = df['Origin'] + ' -> ' + df['Dest']
df['Route'].head()

In [ ]:
# calculate average delay by route
route_delay = df[df['flight_status'] == 'Delayed'].groupby('Route')['DepDelayMinutes'].agg(['mean','count']).query('count >= 100').sort_values('mean', ascending=False).head(10)
route_delay

In [ ]:
# visualize top 10 routes by average delay
route_delay['mean'].plot(kind='barh', color='#9b59b6', edgecolor='white')
plt.title('Top 10 Routes by Average Departure Delay')
plt.xlabel('Avg Delay (Minutes)')
plt.gca().invert_yaxis()
plt.show()

## Key Insights from EDA

1. **Class Imbalance**: Around 75-80% of flights are On Time. Cancelled flights are less than 3%, which means we need to handle class imbalance during model training.

2. **Departure Delay is the Strongest Signal**: DepDelayMinutes and ArrDelayMinutes are highly correlated. A flight delayed at departure is almost always delayed at arrival.

3. **Evening Cascade Effect**: Delay rate increases throughout the day and peaks around 6-8 PM due to accumulated delays from earlier flights.

4. **Seasonal Peaks**: June, July, and December have the highest delay rates due to peak travel seasons.

5. **Airline Variance**: Different airlines have different delay rates, which means airline is a useful predictor.

6. **Right-skewed Delay Distribution**: Most delays are under 60 minutes but extreme outliers exist.